In [1]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

    

import torchvision
import torchvision.transforms as transforms

from models import *   # bring everything in the folder models

global best_prec
use_gpu = torch.cuda.is_available()
print('=> Building model...')
    
    
batch_size = 128

model_name = "VGG16"
#model = cifar10()
model = VGG16()

        

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch, loss_funct):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        # loss = criterion(output, target)

        loss = loss_funct(target, output)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,5)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk) # 5
    batch_size = target.size(0) # 128

    _, pred = output.topk(maxk, 1, True, True) # topk(k, dim=None, largest=True, sorted=True)
                                    # will output (max value, its index)
    pred = pred.t()               # transpose
    correct = pred.eq(target.reshape(1, -1).expand_as(pred))   # "-1": calculate automatically

    res = []
    for k in topk: # 1, 5
        correct_k = correct[:k].reshape(-1).float().sum(0)  # view(-1): make a flattened 1D tensor
        res.append(correct_k.mul_(100.0 / batch_size))   # correct: size of [maxk, batch_size]
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n    ## n is impact factor
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

=> Building model...


/nix/store/sdarj1wic7xmnw0sbw234g3nnk22i28q-python3-3.13.17-env/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [20]:
fdir = 'result/'+str(model_name)

model.load_state_dict(torch.load(os.path.join(fdir, 'checkpoint.pth'), weights_only=True)['state_dict'])

<All keys matched successfully>

In [21]:
criterion = nn.CrossEntropyLoss().cuda()
model = model.cuda()

validate(testloader, model, criterion)

print(abs(model.features[0].weight.detach().cpu()).sum())

Test: [0/79]	Time 0.308 (0.308)	Loss 0.2535 (0.2535)	Prec 93.750% (93.750%)
 * Prec 91.400% 
tensor(204.1767)


In [ ]:
lr = 4e-5
weight_decay = 1e-7
epochs = 30
best_prec = 0

model = model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

if not os.path.exists('result'):
    os.makedirs('result')
    
fdir = 'result/'+str(model_name)

if not os.path.exists(fdir):
    os.makedirs(fdir)

def loss_funct(target, output):
    loss1 = criterion(output, target)
    loss2 = model.features[0].weight.abs().sum()
    return loss1 + loss2

for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch, loss_funct)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))

In [23]:
criterion = nn.CrossEntropyLoss().cuda()
model = model.cuda()

validate(testloader, model, criterion)

print(abs(model.features[0].weight.detach().cpu()).sum())

Test: [0/79]	Time 0.293 (0.293)	Loss 0.4124 (0.4124)	Prec 88.281% (88.281%)
 * Prec 89.270% 
tensor(0.1855)


In [ ]:
fdir = 'result/'+str(model_name)

model.load_state_dict(torch.load(os.path.join(fdir, 'checkpoint.pth'), weights_only=True)['state_dict'])

lr = 4e-5
weight_decay = 1e-7
epochs = 30
best_prec = 0

model = model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

if not os.path.exists('result'):
    os.makedirs('result')
    
fdir = 'result/'+str(model_name)

if not os.path.exists(fdir):
    os.makedirs(fdir)

def loss_funct(target, output):
    loss1 = criterion(output, target)
    loss2 = model.features[0].weight.abs().sum()
    return loss1 + 0.25 * loss2

for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch, loss_funct)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))

In [25]:
criterion = nn.CrossEntropyLoss().cuda()
model = model.cuda()

validate(testloader, model, criterion)

print(abs(model.features[0].weight.detach().cpu()).sum())

Test: [0/79]	Time 0.286 (0.286)	Loss 0.4151 (0.4151)	Prec 92.188% (92.188%)
 * Prec 88.570% 
tensor(0.3976)


In [ ]:
fdir = 'result/'+str(model_name)

model.load_state_dict(torch.load(os.path.join(fdir, 'checkpoint.pth'), weights_only=True)['state_dict'])

lr = 4e-5
weight_decay = 1e-7
epochs = 30
best_prec = 0

model = model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

if not os.path.exists('result'):
    os.makedirs('result')
    
fdir = 'result/'+str(model_name)

if not os.path.exists(fdir):
    os.makedirs(fdir)

def loss_funct(target, output):
    loss1 = criterion(output, target)
    loss2 = model.features[0].weight.abs().sum()
    return loss1 + 0.5 * loss2

for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch, loss_funct)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))

In [27]:
criterion = nn.CrossEntropyLoss().cuda()
model = model.cuda()

validate(testloader, model, criterion)

print(abs(model.features[0].weight.detach().cpu()).sum())

Test: [0/79]	Time 0.328 (0.328)	Loss 0.4467 (0.4467)	Prec 87.500% (87.500%)
 * Prec 87.470% 
tensor(0.2587)


In [ ]:
fdir = 'result/'+str(model_name)

model.load_state_dict(torch.load(os.path.join(fdir, 'checkpoint.pth'), weights_only=True)['state_dict'])

lr = 4e-5
weight_decay = 1e-7
epochs = 30
best_prec = 0

model = model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

if not os.path.exists('result'):
    os.makedirs('result')
    
fdir = 'result/'+str(model_name)

if not os.path.exists(fdir):
    os.makedirs(fdir)

def loss_funct(target, output):
    loss1 = criterion(output, target)
    loss2 = model.features[0].weight.abs().sum()
    return loss1 + 0.75 * loss2

for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch, loss_funct)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))

In [29]:
criterion = nn.CrossEntropyLoss().cuda()
model = model.cuda()

validate(testloader, model, criterion)

print(abs(model.features[0].weight.detach().cpu()).sum())

Test: [0/79]	Time 0.320 (0.320)	Loss 0.3589 (0.3589)	Prec 91.406% (91.406%)
 * Prec 89.620% 
tensor(0.2069)


# Analysis

| Case   | Cost Function      | Test Accuracy (%) | Conv1 Weight Abs Sum |
|--------|--------------------|-------------------|----------------------|
| Y=0    | loss1              | 91.400            | 204.1767             |
| Y=1    | loss1 + loss2      | 89.270            | 0.1855               |
| Y=0.25 | loss1 + 0.25 loss2 | 88.570            | 0.3976               |
| Y=0.50 | loss1 + 0.50 loss2 | 87.470            | 0.2587               |
| Y=0.75 | loss1 + 0.75 loss2 | 89.620            | 0.2069               |

We can see that all of our values are above 80% accuracy. Because of this we can choose any of our Y's. Y=1 is the best choice out of the 5 because it reduces the weight abs sum.